# 03 — Run scenarios

Runs three comparable scenarios on the same agent population and writes per-agent / per-station / per-scenario CSVs plus the required figures.

Scenarios:
1. **S1_real** — real OCM/EIPA charger layout.
2. **S2_clustered** — same number of stations and total ports, clustered tightly near the city centre (radius `SCENARIO.clustered_radius_m`).
3. **S3_distributed** — same number of stations and total ports, on a coarse grid covering the wider study area.

Same RNG seed, same agent count, same horizon → differences in metrics are attributable to layout only.

**Charging-pressure parameters (see `src/config.py → SimulationParams`)** — bumped from the v1 defaults to produce thesis-relevant results:
- horizon: 12 h (was 6 h)
- agents: 500 (was 200)
- trips per agent: 2–4 (was 1–3)
- initial SoC: 0.20–0.55 (was 0.30–0.90)
- low-battery threshold: 0.30 (was 0.20)
- target SoC: 0.70 (was 0.80)
- charger power: 50 kW DC (was 22 kW AC)

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve().parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

from src import config, graph_utils, charger_data, agents as agents_mod, scenarios as scen_mod, simulation as sim_mod, metrics as metrics_mod

print('--- simulation parameters ---')
for k, v in config.SIM.__dict__.items():
    print(f'  {k:30s} = {v}')
print('--- scenario parameters ---')
for k, v in config.SCENARIO.__dict__.items():
    print(f'  {k:30s} = {v}')

In [ ]:
G = graph_utils.get_or_build_graph()
df_clean = charger_data.load_clean_chargers()
print(f'graph: {G.number_of_nodes():,} nodes  | clean chargers: {len(df_clean)} | total ports: {int(df_clean["number_of_points"].sum())}')

In [ ]:
agents = agents_mod.generate_agents(G)
n_trips = sum(len(a.trips) for a in agents)
print(f'generated {len(agents)} agents, total planned trips = {n_trips}')
import numpy as np
soc_array = np.array([a.soc for a in agents])
print(f'initial SoC: mean={soc_array.mean():.2f}  min={soc_array.min():.2f}  max={soc_array.max():.2f}')

In [ ]:
scenarios = scen_mod.build_all_scenarios(df_clean, G)
for s in scenarios:
    print(f'{s.name}: {len(s.stations)} stations | total ports = {s.stations.total_ports()}')

In [ ]:
import copy
# Each scenario gets a fresh deep copy of the agent list so that state
# from one scenario does not leak into the next.
results = []
for s in scenarios:
    fresh_agents = copy.deepcopy(agents)
    sim = sim_mod.Simulator(G, fresh_agents, s.stations, scenario_name=s.name)
    results.append(sim.run(progress=True))

In [ ]:
import pandas as pd
summary_df = pd.DataFrame([r.summary for r in results])
# Reorder for readability.
first_cols = ['scenario', 'n_agents', 'n_stations', 'total_ports',
              'started_charging_events', 'completed_charging_events',
              'total_queued_agents', 'pct_charged_at_least_once',
              'pct_waited_at_least_once']
rest = [c for c in summary_df.columns if c not in first_cols]
summary_df = summary_df[first_cols + rest]
summary_df

In [ ]:
paths = metrics_mod.write_all_outputs(results)
for k, v in paths.items():
    print(f'{k:36s} → {v}')